# Second Models: Four-Model Retrieval Comparison

Run all cells with Internet access and a CUDA GPU. This notebook works on Kaggle, Colab, and local Jupyter using standard Python.

Compare frozen pretrained DINO ViT-S/16, supervised DeiT-S/16, DINOv2 ViT-B/14, and supervised DeiT-B/16 in one session. No training or fine-tuning is performed.

Outputs in `results/02_second_models/`:
- `metrics_validation.csv`
- `metrics_test.csv`
- `runtime_summary.csv`
- `run_metadata.json`

Two ZIP files are saved **next to the repository directory**:
- `second_models_results.zip`: four result files and the shared split manifest.
- `second_models_embeddings.zip`: embeddings for every model and split, cache metadata, run metadata, and the shared split manifest.

Rankings and failure-case figures are not saved by this notebook.

Recorded outputs are preserved from the supplied GPU run. Only project import paths were updated after execution to support the flat `src/` layout.

## 1. Source and experiment configuration

All four models use the same 1,840-image gallery, 1,840 validation queries, and 3,669 test queries across 37 breeds. Relevance means breed agreement. Use the agreed configuration unchanged; the test split has already been evaluated in earlier stages.

In [1]:
from pathlib import Path
import datetime as dt
import hashlib
import json
import os
import shlex
import subprocess
import sys
import uuid

REPOSITORY_URL = "https://github.com/kite121/Visual_retrieval_system.git"
BRANCH = "main"
MODELS = [
    "dino_vit_s16",
    "deit_vit_s16_supervised",
    "dinov2_vit_b14",
    "deit_vit_b16_supervised",
]
BATCH_SIZE = 64
NUM_WORKERS = 2
TIMING_REPEATS = 3
EXPECTED_MANIFEST_SHA256 = "82f40e25888189aba46d1c1ff2c74894284c1ff7fa50362026c8abbea0ec9f80"

def run(command):
    command = [str(part) for part in command]
    print("$", shlex.join(command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, check=True)

start_dir = Path.cwd().resolve()
local_repositories = [
    parent for parent in (start_dir, *start_dir.parents)
    if (parent / ".git").is_dir()
    and (parent / "scripts" / "run_evaluation.py").is_file()
]
if local_repositories:
    REPO_DIR = local_repositories[0]
else:
    REPO_DIR = start_dir / "Visual_retrieval_system"
    if not REPO_DIR.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--depth", "1",
            REPOSITORY_URL, str(REPO_DIR),
        ], check=True)
    else:
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout")
        run(["git", "checkout", BRANCH])
        run(["git", "pull", "--ff-only", "origin", BRANCH])

os.chdir(REPO_DIR)
SOURCE_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True,
).strip()
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
RESULT_DIR = REPO_DIR / "results" / "02_second_models"
ARTIFACT_DIR = REPO_DIR / "artifacts" / "02_second_models" / RUN_ID
TIMING_DIR = ARTIFACT_DIR / "timing_measurements"
print("Repository:", REPO_DIR)
print("Source commit:", SOURCE_COMMIT)
print("Run ID:", RUN_ID)

Cloning into '/kaggle/working/Visual_retrieval_system'...


Repository: /kaggle/working/Visual_retrieval_system
Source commit: d514f45ca1878e27fb9d01cf2eed774e82c40a45
Run ID: 20261107T180431Z_a18259


## 2. Dependencies and GPU

Install the single root dependency file. Existing compatible PyTorch and torchvision installations are retained. The full four-model evaluation requires a CUDA GPU. A smaller batch size can be selected in the configuration cell before starting.

In [2]:
run([
    sys.executable, "-m", "pip", "install",
    "--disable-pip-version-check", "-q", "-r", "requirements.txt",
])

probe = subprocess.check_output([
    sys.executable, "-c",
    "import json,sys,torch; sys.path.insert(0,'.'); "
    "from src import models,timing; "
    "assert torch.cuda.is_available(), 'Enable a CUDA GPU before running this notebook'; "
    "print(json.dumps({'hardware':timing.hardware_info('cuda'),"
    "'registered_models':models.available_models()}))",
], cwd=REPO_DIR, text=True)
environment = json.loads(probe)
assert set(MODELS).issubset(environment["registered_models"])
DEVICE = "cuda"
print(json.dumps(environment["hardware"], indent=2))

help_text = subprocess.check_output([
    sys.executable, "scripts/run_evaluation.py", "--help",
], cwd=REPO_DIR, text=True)
assert "--no-save-rankings" in help_text and "--experiment-id" in help_text, (
    "Update the repository checkout to the current main branch."
)

$ /usr/bin/python3 -m pip install --disable-pip-version-check -q -r requirements.txt


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


{
  "device": "cuda",
  "python": "3.13.16",
  "torch": "2.11.0+cu130",
  "platform": "Linux-6.18.55+-x86_64-with-glibc2.39",
  "cpu_count": 4,
  "gpu": "Tesla T4",
  "cuda": "13.0"
}


## 3. Download data and verify the shared split

Reconstruct the seed-42 split and compare it with the committed manifest. The manifest is not replaced. Both query splits must be disjoint from the gallery.

In [3]:
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

run([sys.executable, "scripts/prepare_dataset.py", "--check"])
manifest_path = REPO_DIR / "configs" / "split_manifest.csv"
manifest_bytes = manifest_path.read_bytes()
assert hashlib.sha256(manifest_bytes).hexdigest() == EXPECTED_MANIFEST_SHA256
manifest = pd.read_csv(manifest_path)
assert manifest["split"].value_counts().to_dict() == {
    "gallery": 1840, "val": 1840, "test": 3669,
}
assert manifest["image_id"].is_unique and manifest["breed"].nunique() == 37
display(manifest.groupby("split").agg(
    images=("image_id", "size"), breeds=("breed", "nunique"),
).reindex(["gallery", "val", "test"]))

$ /usr/bin/python3 scripts/prepare_dataset.py --check


100%|██████████| 792M/792M [00:16<00:00, 47.0MB/s] 
100%|██████████| 19.2M/19.2M [00:01<00:00, 18.4MB/s]


images: 7349  breeds: 37
gallery: 1840  val: 1840  test: 3669
manifest is reproducible


,images,breeds
split,,
gallery,1840,37
val,1840,37
test,3669,37


## 4. Quality evaluation

Run all four models in one invocation so the canonical files contain all four rows. Extract fresh gallery, validation, and test embeddings; calculate Precision@5/10, Recall@5/10, and Hit@5. Similarity search is used to compute metrics, but ranking arrays are not written.

The initial single-run runtime table is replaced in the next step by the three-repeat summary.

In [4]:
run([
    sys.executable, "scripts/run_evaluation.py",
    "--models", *MODELS,
    "--device", DEVICE,
    "--batch-size", BATCH_SIZE,
    "--num-workers", NUM_WORKERS,
    "--results-dir", RESULT_DIR,
    "--artifacts", ARTIFACT_DIR,
    "--experiment-id", "02_second_models",
    "--output-format", "canonical",
    "--no-save-rankings",
    "--rebuild",
])

quality_runs = [
    json.loads((ARTIFACT_DIR / "runs" / model / "evaluation.json").read_text())
    for model in MODELS
]
expected_timing_ids = quality_runs[0]["query_latency"]["query_ids"]
assert len(expected_timing_ids) == len(set(expected_timing_ids)) == 200
for result in quality_runs:
    assert result["pretrained"] and not result["gallery_build_cache_reused"]
    assert result["manifest_sha256"] == EXPECTED_MANIFEST_SHA256
    assert result["sizes"] == {"gallery": 1840, "val": 1840, "test": 3669}
    assert result["query_latency"]["query_ids"] == expected_timing_ids
    assert result["hardware"] == environment["hardware"]
    assert set(result["metrics"]) == {"val", "test"}
    for split in ("val", "test"):
        assert set(result["metrics"][split]) == {"P@5", "P@10", "R@5", "R@10", "Hit@5"}
        assert all(np.isfinite(v) and 0 <= v <= 1 for v in result["metrics"][split].values())
assert not list(ARTIFACT_DIR.rglob("topk_*.npz"))
print("Quality evaluation complete for all four models.")

$ /usr/bin/python3 scripts/run_evaluation.py --models dino_vit_s16 deit_vit_s16_supervised dinov2_vit_b14 deit_vit_b16_supervised --device cuda --batch-size 64 --num-workers 2 --results-dir /kaggle/working/Visual_retrieval_system/results/02_second_models --artifacts /kaggle/working/Visual_retrieval_system/artifacts/02_second_models/20261107T180431Z_a18259 --experiment-id 02_second_models --output-format canonical --no-save-rankings --rebuild

=== DINO ViT-S/16 (dino_vit_s16) on cuda ===
Downloading: "https://github.com/facebookresearch/dino/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_deitsmall16_pretrain.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 519MB/s]


gallery: (1840, 384), build time 8.0 s


 val: P@5=0.8609  R@5=0.0866  P@10=0.8278  R@10=0.1665  Hit@5=0.9826


test: P@5=0.8416  R@5=0.0847  P@10=0.8156  R@10=0.1641  Hit@5=0.9747
query latency (median): embed 5.98 ms, search 0.218 ms, preprocess 5.6 ms

=== DeiT-S/16 (supervised) (deit_vit_s16_supervised) on cuda ===


gallery: (1840, 384), build time 7.8 s


... (8 outputs left out)

embedding:   0%|          | 0/29 [00:00<?, ?it/s]         

gallery: (1840, 768), build time 20.6 s


 val: P@5=0.8939  R@5=0.0899  P@10=0.8767  R@10=0.1764  Hit@5=0.9875


test: P@5=0.8752  R@5=0.0880  P@10=0.8604  R@10=0.1731  Hit@5=0.9678
query latency (median): embed 12.27 ms, search 0.243 ms, preprocess 5.7 ms

wrote tables to /kaggle/working/Visual_retrieval_system/results/02_second_models
Quality evaluation complete for all four models.


## 5. Repeated runtime measurements

Measure the same 200 query IDs three times per model. Rebuild gallery features for each repeat, rotate model order, and use GPU synchronization and warm-up.

Only `runtime_summary.csv` is exported. Its latency columns contain the median, minimum, and maximum **of the three per-repeat medians**; gallery columns contain the median, minimum, and maximum of the three gallery build times. They are not confidence intervals or per-query extrema. Raw timing records remain local for validation.

In [5]:
run([
    sys.executable, "scripts/benchmark_timing.py",
    "--models", *MODELS,
    "--device", DEVICE,
    "--batch-size", BATCH_SIZE,
    "--num-workers", NUM_WORKERS,
    "--repeats", TIMING_REPEATS,
    "--output-dir", TIMING_DIR,
    "--summary-dir", RESULT_DIR,
])

timing_records = []
for model in MODELS:
    for repeat in range(1, TIMING_REPEATS + 1):
        path = TIMING_DIR / f"{model}_repeat{repeat}.json"
        record = json.loads(path.read_text())
        assert record["model"] == model and record["repeat"] == repeat
        assert record["pretrained"] and not record["gallery_build_cache_reused"]
        assert record["manifest_sha256"] == EXPECTED_MANIFEST_SHA256
        assert record["query_latency"]["query_ids"] == expected_timing_ids
        assert record["hardware"] == environment["hardware"]
        assert record["batch_size"] == BATCH_SIZE
        assert record["num_workers"] == NUM_WORKERS
        timing_records.append(record)

runtime_summary = pd.read_csv(RESULT_DIR / "runtime_summary.csv")
assert set(runtime_summary["model_id"]) == set(MODELS)
assert runtime_summary["model_id"].is_unique
assert (runtime_summary["repeats"] == TIMING_REPEATS).all()
for model in MODELS:
    row = runtime_summary.set_index("model_id").loc[model]
    records = [r for r in timing_records if r["model"] == model]
    for kind in ("embed", "search", "preprocess"):
        values = [r["query_latency"][f"{kind}_ms"]["median"] for r in records]
        for label, operation in (("median", np.median), ("min", np.min), ("max", np.max)):
            assert np.isclose(row[f"{kind}_{label}_ms"], operation(values))
    values = [r["gallery_build_seconds"] for r in records]
    for label, operation in (("median", np.median), ("min", np.min), ("max", np.max)):
        assert np.isclose(row[f"gallery_build_seconds_{label}"], operation(values))
print("Verified 12 measurements and four summary rows.")

$ /usr/bin/python3 scripts/benchmark_timing.py --models dino_vit_s16 deit_vit_s16_supervised dinov2_vit_b14 deit_vit_b16_supervised --device cuda --batch-size 64 --num-workers 2 --repeats 3 --output-dir /kaggle/working/Visual_retrieval_system/artifacts/02_second_models/20261107T180431Z_a18259/timing_measurements --summary-dir /kaggle/working/Visual_retrieval_system/results/02_second_models
repeat 1/3: dino_vit_s16


Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main


saved dino_vit_s16: gallery 8.14s, embed 6.20ms
repeat 1/3: deit_vit_s16_supervised


saved deit_vit_s16_supervised: gallery 7.98s, embed 5.03ms
repeat 1/3: dinov2_vit_b14


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_7764ea0f912e53c92e82eb78a2a1631e92725fc8
/root/.cache/torch/hub/facebookresearch_dinov2_7764ea0f912e53c92e82eb78a2a1631e92725fc8/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_7764ea0f912e53c92e82eb78a2a1631e92725fc8/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_7764ea0f912e53c92e82eb78a2a1631e92725fc8/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


saved dinov2_vit_b14: gallery 28.68s, embed 18.36ms
repeat 1/3: deit_vit_b16_supervised


... (9 outputs left out)

saved deit_vit_b16_supervised: gallery 20.69s, embed 12.47ms
repeat 3/3: dino_vit_s16


Using cache found in /root/.cache/torch/hub/facebookresearch_dino_main


saved dino_vit_s16: gallery 8.32s, embed 6.25ms
repeat 3/3: deit_vit_s16_supervised


saved deit_vit_s16_supervised: gallery 7.76s, embed 5.10ms
completed 12 timing measurements: /kaggle/working/Visual_retrieval_system/artifacts/02_second_models/20261107T180431Z_a18259/timing_measurements
Verified 12 measurements and four summary rows.


## 6. Complete run metadata and inspect results

Record the Git commit, actual installed package versions, precise timing aggregation, and the common latency-query sample. The four result files contain the same experiment identifier and hardware context.

In [6]:
from importlib.metadata import version

metadata_path = RESULT_DIR / "run_metadata.json"
metadata = json.loads(metadata_path.read_text())
assert metadata["experiment_id"] == "02_second_models"
metadata["run_id"] = RUN_ID
metadata["source_commit"] = SOURCE_COMMIT
metadata["completed_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
metadata["environment"]["packages"] = {
    package: version(package)
    for package in (
        "torch", "torchvision", "timm", "numpy", "pandas",
        "scikit-learn", "Pillow", "PyYAML",
    )
}
metadata["timing"].update({
    "repeats_per_model": TIMING_REPEATS,
    "query_ids": expected_timing_ids,
    "model_order": "rotated between repeats",
    "runtime_summary_source": "three fresh measurements per model after quality evaluation",
    "latency_aggregation": "median/min/max of the per-repeat median latency",
    "gallery_aggregation": "median/min/max of the per-repeat gallery build time",
})
metadata["artifacts"] = {
    "rankings_saved": False,
    "embeddings_archive": "second_models_embeddings.zip",
    "embedding_splits": ["gallery", "val", "test"],
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")

validation_metrics = pd.read_csv(RESULT_DIR / "metrics_validation.csv")
test_metrics = pd.read_csv(RESULT_DIR / "metrics_test.csv")
for frame in (validation_metrics, test_metrics):
    assert frame["model_id"].is_unique and set(frame["model_id"]) == set(MODELS)
    values = frame.drop(columns=["model_id", "model"]).to_numpy()
    assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()

print("Validation metrics")
display(validation_metrics.round(4))
print("Test metrics")
display(test_metrics.round(4))
print("Repeated runtime summary")
display(runtime_summary.round(3))

Validation metrics


,model_id,model,precision_at_5,recall_at_5,precision_at_10,recall_at_10,hit_at_5
0,dino_vit_s16,DINO ViT-S/16,0.8609,0.0866,0.8278,0.1665,0.9826
1,deit_vit_s16_supervised,DeiT-S/16 (supervised),0.8883,0.0893,0.8692,0.1749,0.9859
2,dinov2_vit_b14,DINOv2 ViT-B/14,0.9112,0.0916,0.8884,0.1787,0.9848
3,deit_vit_b16_supervised,DeiT-B/16 (supervised),0.8939,0.0899,0.8767,0.1764,0.9875


Test metrics


,model_id,model,precision_at_5,recall_at_5,precision_at_10,recall_at_10,hit_at_5
0,dino_vit_s16,DINO ViT-S/16,0.8416,0.0847,0.8156,0.1641,0.9747
1,deit_vit_s16_supervised,DeiT-S/16 (supervised),0.8665,0.0872,0.8527,0.1716,0.9692
2,dinov2_vit_b14,DINOv2 ViT-B/14,0.9064,0.0912,0.8881,0.1787,0.9820
3,deit_vit_b16_supervised,DeiT-B/16 (supervised),0.8752,0.0880,0.8604,0.1731,0.9678


Repeated runtime summary


,model_id,repeats,gallery_build_seconds_median,gallery_build_seconds_min,gallery_build_seconds_max,embed_median_ms,embed_min_ms,embed_max_ms,search_median_ms,search_min_ms,search_max_ms,preprocess_median_ms,preprocess_min_ms,preprocess_max_ms,n_queries,search_k,device,gpu,batch_size,num_workers
0,dino_vit_s16,3,8.142,8.137,8.319,6.253,6.198,6.297,0.224,0.221,0.225,5.596,5.529,5.603,200,5,cuda,Tesla T4,64,2
1,deit_vit_s16_supervised,3,7.830,7.761,7.983,5.101,5.028,5.325,0.226,0.222,0.227,5.583,5.539,5.708,200,5,cuda,Tesla T4,64,2
2,dinov2_vit_b14,3,28.158,28.151,28.679,18.638,18.364,18.796,0.252,0.250,0.258,5.638,5.618,5.664,200,5,cuda,Tesla T4,64,2
3,deit_vit_b16_supervised,3,20.618,20.409,20.694,12.470,12.429,12.503,0.253,0.247,0.255,5.664,5.613,5.703,200,5,cuda,Tesla T4,64,2


## 7. Compare quality with the historical four-model experiment

Compare every quality value against the committed Stage 3 results. A difference is reported explicitly. The historical values remain unchanged, and timing differences are expected when hardware or environment changes.

In [7]:
historical = pd.read_csv(REPO_DIR / "results" / "stage3" / "quality" / "metrics.csv")
historical = historical.set_index(["model", "split"])
metric_names = {
    "precision_at_5": "P@5", "recall_at_5": "R@5",
    "precision_at_10": "P@10", "recall_at_10": "R@10",
    "hit_at_5": "Hit@5",
}
comparison_rows = []
for split, frame in (("val", validation_metrics), ("test", test_metrics)):
    for row in frame.to_dict("records"):
        old = historical.loc[(row["model"], split)]
        for column, old_column in metric_names.items():
            comparison_rows.append({
                "model_id": row["model_id"], "split": split,
                "metric": column, "previous": float(old[old_column]),
                "current": float(row[column]),
                "absolute_difference": abs(float(row[column]) - float(old[old_column])),
            })
comparison = pd.DataFrame(comparison_rows)
assert len(comparison) == 40
max_difference = float(comparison["absolute_difference"].max())
print("Compared quality values:", len(comparison))
print("Maximum absolute difference:", max_difference)
if max_difference <= 1e-6:
    print("Quality metrics match the historical experiment within 1e-6.")
else:
    print("Quality differs: review these rows before replacing historical results.")
    display(comparison.loc[comparison["absolute_difference"] > 1e-6])
metadata["historical_comparison"] = {
    "source": "results/stage3/quality/metrics.csv",
    "values_compared": len(comparison),
    "max_absolute_difference": max_difference,
    "tolerance": 1e-6,
    "within_tolerance": max_difference <= 1e-6,
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")

Compared quality values: 40
Maximum absolute difference: 0.0
Quality metrics match the historical experiment within 1e-6.


8822

## 8. Validate embeddings

Require complete image-ID and label coverage, finite L2-normalized vectors, and the expected feature dimension for every model and split. The four models produce 12 embedding arrays in total.

In [8]:
dimension_by_model = {
    "dino_vit_s16": 384,
    "deit_vit_s16_supervised": 384,
    "dinov2_vit_b14": 768,
    "deit_vit_b16_supervised": 768,
}
embedding_files = []
for model in MODELS:
    for split in ("gallery", "val", "test"):
        expected = manifest.loc[manifest["split"] == split].sort_values("image_id", kind="stable")
        cache_dir = ARTIFACT_DIR / "embeddings" / model
        array_path = cache_dir / f"{split}.npz"
        info_path = cache_dir / f"{split}.json"
        with np.load(array_path, allow_pickle=False) as arrays:
            vectors = arrays["embeddings"]
            assert arrays["image_ids"].tolist() == expected["image_id"].tolist()
            assert arrays["labels"].tolist() == expected["label"].tolist()
            assert vectors.shape == (len(expected), dimension_by_model[model])
            assert np.isfinite(vectors).all()
            assert np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-3)
        cache_metadata = json.loads(info_path.read_text())
        assert cache_metadata["pretrained"] and not cache_metadata["cache_reused"]
        assert cache_metadata["n_images"] == len(expected)
        embedding_files.extend([array_path, info_path])
assert len(embedding_files) == 24
print("Verified 12 embedding arrays and 12 cache metadata files.")

Verified 12 embedding arrays and 12 cache metadata files.


## 9. Export two separate ZIP files

Both ZIP files are created in the parent directory of the repository. Embeddings are stored as feature arrays; pretrained model weight files and the dataset are not bundled. The ZIP contents are checked against the source files before download.

In [9]:
import zipfile

expected_results = {
    "metrics_validation.csv", "metrics_test.csv",
    "runtime_summary.csv", "run_metadata.json",
}
assert {p.name for p in RESULT_DIR.iterdir()} == expected_results
results_archive = REPO_DIR.parent / "second_models_results.zip"
embeddings_archive = REPO_DIR.parent / "second_models_embeddings.zip"

def write_and_verify_archive(path, entries):
    checksums = {}
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as archive:
        for source, archive_name in entries:
            archive.write(source, archive_name)
            checksums[archive_name] = hashlib.sha256(source.read_bytes()).hexdigest()
    with zipfile.ZipFile(path) as archive:
        assert archive.testzip() is None
        assert set(archive.namelist()) == set(checksums)
        for archive_name, expected_hash in checksums.items():
            assert hashlib.sha256(archive.read(archive_name)).hexdigest() == expected_hash
    print(f"Created {path} ({path.stat().st_size / 1024**2:.1f} MiB)")
    display(FileLink(os.path.relpath(path, Path.cwd())))

common_entries = [
    (manifest_path, "configs/split_manifest.csv"),
]
write_and_verify_archive(results_archive, [
    *common_entries,
    *[(RESULT_DIR / name, f"results/02_second_models/{name}") for name in sorted(expected_results)],
])
write_and_verify_archive(embeddings_archive, [
    *common_entries,
    (metadata_path, "results/02_second_models/run_metadata.json"),
    *[(path, "artifacts/02_second_models/" + path.relative_to(ARTIFACT_DIR).as_posix())
      for path in embedding_files],
])

Created /kaggle/working/second_models_results.zip (0.0 MiB)


/kaggle/working/second_models_results.zip

Created /kaggle/working/second_models_embeddings.zip (60.1 MiB)


/kaggle/working/second_models_embeddings.zip

## Completion

Download both ZIP files before ending the session. The result ZIP is intended for review and Git publication. The embeddings ZIP can be stored separately and reused for the demo or a teammate's later analysis.

The notebook reports any differences from the historical quality results. Review them before changing report numbers. Failure-case selection and ranking export are handled separately by the teammate responsible for that work.